In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth agno lancedb sentence-transformers pillow pandas tqdm
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.0
!pip install --no-deps trl==0.22.2

In [5]:
!pip install agno lancedb sentence-transformers pillow pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.2/39.2 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 21.1 MB/s eta 0:00:00


In [14]:
import nest_asyncio
nest_asyncio.apply()  # ✅ MUST be before importing agno


In [10]:
from unsloth import FastVisionModel
from agno.agent import Agent
from agno.db.sqlite import SqliteDb
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.embedder.sentence_transformer import SentenceTransformerEmbedder
from agno.models.openai import OpenAIChat
from agno.vectordb.lancedb import LanceDb
from agno.tools.knowledge import KnowledgeTools
import torch
import pandas as pd
import os
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
import json
import base64
from io import BytesIO

/tmp/ipython-input-2646005515.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastVisionModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 12-05 10:56:17 [fa_utils.py:72] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


In [11]:
political_kb_content = """
# Q: Who is Sheikh Hasina?

A: Sheikh Hasina Wazed is the former Prime Minister of Bangladesh who served from 2009 to 2024. She is the leader of the Awami League party, whose symbol is a boat (nouka). She is the daughter of Sheikh Mujibur Rahman, the founder of Bangladesh. Common names: Hasina, Sheikh Hasina, PM Hasina. Party colors: red and green.

Q: What is the symbol of Awami League?
A: The symbol of Awami League is a boat (nouka in Bengali). It appears on a red background and is the party's official election symbol. If you see a boat symbol in a meme, it's likely political and related to Awami League.

Q: Who is Khaleda Zia?
A: Begum Khaleda Zia is the former Prime Minister of Bangladesh and current chairperson of the Bangladesh Nationalist Party (BNP). She served as PM from 1991-1996 and 2001-2006. The BNP symbol is a sheaf of paddy (rice). She is the widow of President Ziaur Rahman.

Q: What does "Joy Bangla" mean?
A: "Joy Bangla" means "Victory to Bengal" in English. It is the national slogan of Bangladesh and strongly associated with the Awami League party and the 1971 Liberation War. If you see this slogan in a meme, it indicates political content related to Awami League or nationalist sentiment.

Q: What is the BNP party symbol?
A: The BNP (Bangladesh Nationalist Party) symbol is a sheaf of paddy (dhan er sheesh), which represents rice stalks. It appears on a green background. If you see this symbol, the content is political and related to BNP.

Q: What happened in the 2024 Bangladesh protests?
A: In July-August 2024, massive student-led protests occurred in Bangladesh over the quota system in government jobs. These protests led to the resignation of Prime Minister Sheikh Hasina on August 5, 2024. Keywords: quota reform, student movement, August 2024. Any meme referencing these events is highly political.

Q: How do I identify Awami League content?
A: Look for: boat symbol, red and green colors together, Sheikh Hasina's image, Sheikh Mujib's photo, "Joy Bangla" slogan, references to Liberation War 1971, or any content about the current/former government.

Q: How do I identify BNP content?
A: Look for: sheaf of paddy symbol, green color dominant, Khaleda Zia or Tarique Rahman's images, "Bangladesh Zindabad" slogan, opposition movement references, or anti-government content from 2009-2024 period.

similar to this, create an extensive Q and A knowledgebase of all recent political events in Bangladesh and Overseas, especially america. For bangladesh, include political student leagues, new parties formed and political events after August 5th 2024 (when Sheikh hasina fled the country)

Sheikh Hasina’s ouster in August 2024 triggered a wave of new parties, student-led formations, and shifting alignments in Bangladesh, while the United States saw Donald Trump return to the presidency after the 2024 election.  Below is a Q\&A-style knowledge base in the same style as your examples, focused on post‑August‑5‑2024 Bangladesh politics and recent US politics.

***

## Bangladesh core events after August 5, 2024

**Q: What happened in Bangladesh on August 5, 2024?**
A: On August 5, 2024, Prime Minister Sheikh Hasina stepped down and fled the country after weeks of massive, student‑led protests against the quota system and authoritarian governance, marking the end of around 15 years of Awami League rule.  This turning point is often called the July Revolution or July–August 2024 uprising.

**Q: What is the “July Revolution 2024” in Bangladesh?**
A: The July Revolution 2024 refers to the student‑driven uprising that began with quota reform protests and escalated into a nationwide anti‑autocracy movement, culminating in the fall of Sheikh Hasina’s government in early August.  It combined campus‑based mobilization, civil society support, and social‑media‑driven “citizen journalism” to bypass state‑controlled narratives.

***

## Interim government and Muhammad Yunus

**Q: Who governs Bangladesh after Sheikh Hasina’s fall?**
A: After Hasina’s ouster, an interim administration headed by Nobel laureate Dr. Muhammad Yunus was installed as a caretaker authority to oversee reforms and prepare for elections.  This interim government promised to end political persecution, tackle corruption, and initiate legal and institutional reforms.

**Q: Has the interim government in Bangladesh delivered on its reform promises?**
A: Analyses argue that while the interim government raised hopes of a “new beginning,” Bangladesh remains trapped in cycles of revenge justice, mob violence, and weak institutions, suggesting only superficial change so far.  Structural legacies of repression and patronage politics continue to limit meaningful democratic consolidation.

***

## Student movements and political student leagues

**Q: What role did students play in the 2024 Bangladesh uprising?**
A: Students were the primary organizers and frontline participants, transforming quota reform protests into a broad anti‑autocracy movement that united diverse groups and eventually toppled the government.  University campuses, especially Dhaka University, became hubs for organizing marches, strikes, and online campaigns.

**Q: What is the Bangladesh Chhatra League, and how was it involved?**
A: The Bangladesh Chhatra League (BCL) is the student wing of the Awami League, historically used to mobilize support and sometimes to confront opposition on campuses.  During the 2024 quota movement, BCL activists were reported to have clashed violently with protesting students, reinforcing perceptions of regime‑aligned campus militias.

**Q: How did social media and “citizen journalism” influence the 2024 student movement?**
A: Protesters used platforms like Facebook and other social media to share live footage, coordinate actions, and counter state narratives, building on earlier digital activism in 2013 and 2018.  Hashtags, red profile pictures, and grassroots livestreaming helped nationalize the movement and internationalize awareness of the crackdown.

***

## New parties and post‑Hasina party landscape

**Q: How many new political parties emerged in Bangladesh after August 2024?**
A: Within about nine months after the uprising, 24 new political parties were formed, and 65 new outfits applied for registration with the Election Commission, in addition to around 55 already registered parties.  The burst of party creation reflects both the political opening and fragmentation in post‑Hasina Bangladesh.
**Q: What is the Nucleus Party of Bangladesh (NPB)?**
A: The Nucleus Party of Bangladesh (NPB) was among the first new groups launched after the uprising, announced on August 23, 2024, near Dhaka University’s central library.  Early commentary describes NPB as lacking a clear ideological profile and emerging more as a reaction to the old order than as a fully articulated programmatic party.

**Q: What is the National Citizen(s) Party (NCP) in Bangladesh?**
A: The National Citizen(s) Party (NCP) is a youth‑ and student‑led political party formally launched on February 28, 2025, by leaders of the 2024 uprising, positioning itself as a centrist, anti‑duopoly alternative to Awami League and BNP.  NCP’s convenor and main public face is Nahid Islam, a prominent protest leader who previously advised the Yunus‑led interim government.

**Q: What are the goals and slogans of the NCP?**
A: NCP leaders frame the 2024 uprising as the start of a “second republic” and call for a new democratic constitution that prevents future one‑party autocracy.  The party emphasizes youth representation, anti‑nepotism, institutional reform, and “agenda‑driven politics” focused on jobs, economic development, and national unity.

**Q: How electorally successful is NCP so far?**
A: Reporting indicates that, despite drawing large crowds, NCP has struggled to convert street energy into votes, polling around third place nationally and failing to win key student union elections at Dhaka University.  Structural disadvantages include weak rural networks, limited funding, and competition with entrenched BNP and Islamist parties courting the same youth vote.

**Q: Are student leaders forming other parties besides NCP?**
A: Yes, multiple student and youth platforms that led the July–August uprising have discussed or initiated party projects, announcing plans by late 2024 to register a new party and contest the next general election.  These initiatives generally claim a non‑left, non‑right, issue‑driven profile, though few have yet built nationwide organization on the scale of NCP.

***

## Traditional parties after 2024 (AL, BNP, Jamaat, others)

**Q: What happened to the Awami League after the July 2024 revolution?**
A: The Awami League lost power when Hasina’s government fell and is currently barred from participating in upcoming elections under the terms set by the interim authorities.  Party figures warn that excluding AL could trigger unrest, especially given its deep ties to the public sector and the garment‑export industry.
**Q: What is the status of the Bangladesh Nationalist Party (BNP) after 2024?**
A: BNP, long the main opposition, is part of discussions with the interim government on electoral reforms and is seeking to benefit from anti‑AL sentiment while also trying to appeal to the youth.  BNP leaders highlight the importance of integrating young activists into parliament and forming alliances with new and Islamist‑leaning parties.

**Q: Where does Jamaat‑e‑Islami Bangladesh fit in post‑Hasina politics?**
A: Academic work notes that fears of Jamaat’s potential rise were used by the former regime and some regional actors to discredit the student uprising, though the movement itself was broad‑based and not led by Jamaat.  Despite bans and repression over the previous decade and a half, Jamaat and Islamist currents remain resilient and are part of the wider debate on the future political order.

**Q: Are Sufi‑oriented or religious parties important in the 2024 revolution?**
A: Research highlights that Sufi‑influenced and Islamic organizations, including groups like Hefazat‑e‑Islam, interacted with and sometimes supported protests against the Awami League, contributing to the broader July Revolution context.  Their role underscores the mix of secular, left, nationalist, and religious actors within the anti‑regime coalition.

***

## Political violence, justice, and “second republic” debates

**Q: What kinds of violence and abuses have been reported since the regime change?**
A: Analyses one year on describe patterns of revenge justice, mob attacks, and targeting of perceived regime loyalists, alongside continuing religious extremism and weak rule of law.  These dynamics raise concerns that transitional justice is being pursued informally and violently rather than through robust legal mechanisms.

**Q: What is the “July National Charter” in Bangladesh?**
A: The July National Charter is a political declaration emerging from the 2024 revolution that outlines demands for reforms and a new governance framework rooted in popular sovereignty.  Some parties and activists are pushing for its legal validation via a plebiscite or constitutional order to embed revolutionary goals within formal law.

**Q: Why are scholars talking about a “second republic” in Bangladesh?**
A: Activists and new parties argue that Bangladesh needs a foundational reset—a second republic—to prevent future authoritarianism, update constitutional principles, and reflect the aspirations of the 2024 uprising.  This debate connects back to Sheikh Mujibur Rahman’s original vision of nationalism, secularism, socialism, and democracy, and how those ideals can be reinterpreted today.

***

## Bangladesh: recent youth and civil society dynamics

**Q: How has civil society contributed to political change in 2024 Bangladesh?**
A: Civil society organizations, rights groups, and professional associations helped coordinate protests, bridge divides between social groups, and amplify student demands nationally and internationally.  Despite attempts at co‑optation and repression, their sustained mobilization weakened the regime’s moral authority and supported the revolution’s success.

**Q: How are young people’s political attitudes changing in Bangladesh?**
A: Earlier research already showed youth frustration with elite‑dominated, patronage‑based politics and skepticism toward traditional parties, trends that fed into the 2024 uprising.  Post‑uprising developments, including NCP’s creation and student‑led initiatives, demonstrate a push for more direct youth representation and issue‑based politics.
***

## United States: recent political events

**Q: Who is the current president of the United States, and how did he return to power?**
A: Donald Trump returned to the US presidency after winning the November 2024 election against Democratic nominee Vice President Kamala Harris.  He secured at least 277 Electoral College votes and a narrow but clear popular‑vote lead, framing the result as a mandate to reverse Biden‑Harris era policies.

**Q: What were key issues in the 2024 US presidential election?**
A: The contest was widely viewed as a referendum on the Biden‑Harris administration, with debates centered on inflation, immigration, crime, foreign policy, and culture‑war topics.  Analysts note that Trump’s victory reflected voter dissatisfaction with incumbents more than broad enthusiasm for his detailed policy agenda.

**Q: How did Trump’s 2024 victory compare to 2016 and 2020?**
A: Commentators emphasize that Trump improved his raw vote total compared to 2020 but still fell short of Joe Biden’s 2020 number, winning this time because the Democratic coalition was weaker under Harris and pivotal swing‑state margins shifted.  The outcome again hinged on relatively small vote shifts in states like Michigan, Pennsylvania, and Wisconsin.

**Q: What are current US–Bangladesh political linkages or concerns?**
A: International coverage of the 2024 Bangladesh uprising highlighted concerns about democratic backsliding, human rights, and regional stability, themes closely watched in US foreign policy circles.  Post‑uprising, Western actors, including the US, have focused on the conduct of the interim government, electoral reforms, and treatment of political opponents.

***


## BNP (Bangladesh Nationalist Party) – Key Figures

**Q: Who is Tarique Rahman?**
A: Tarique Rahman (born November 20, 1965/1967) is the acting chairman of the Bangladesh Nationalist Party (BNP) and son of former President Ziaur Rahman and former Prime Minister Khaleda Zia.  He has been leading the party from exile in London since 2018 after his mother was imprisoned.  Common names: Tarique Zia, Acting Chairman Tarique, Zarek Tia(Sarcastic Name)

**Q: What is Tarique Rahman's current political status?**
A: After Sheikh Hasina's fall in August 2024, Tarique Rahman's political stature has grown significantly, with Interim Chief Adviser Muhammad Yunus meeting him in London in June 2025 to discuss election dates.  He has pledged to return to Bangladesh once legal cases against him are lifted and to support the interim government's reform process.

**Q: How does Tarique Rahman connect with young voters?**
A: Tarique Rahman faces the challenge of connecting BNP with Gen Z voters who led the 2024 uprising, as many young people criticize the party due to allegations of crime and extortion by some BNP activists.  BNP insiders say he will return after the election date is announced to campaign directly.

**Q: Who is Salahuddin Ahmed in BNP?**
A: Salahuddin Ahmed is a BNP Standing Committee member who has publicly stated that party acting chairman Tarique Rahman will return to Bangladesh soon.  He is among senior BNP leaders coordinating with the interim government on electoral reforms.


***

## Awami League – Key Figures

**Q: Who is Obaidul Quader?**
A: Obaidul Quader (born February 1, 1952) is the General Secretary of Bangladesh Awami League, a position he has held since October 2016 for three consecutive terms.  He previously served as Minister of Road Transport and Bridges from 2011–2024 and represented the Noakhali-5 constituency in parliament from 2009–2024.

**Q: What happened to Obaidul Quader after August 2024?**
A: Following the fall of the Hasina government, Obaidul Quader's status became unclear, with reports describing him as exiled.  He was a key figure in the Awami League government and served as the party's media spokesperson for years.

**Q: Who is Shajahan Khan?**
A: Shajahan Khan (born January 1, 1952) is a former eight-term member of parliament from Madaripur-2 constituency (1986–2024) and presidium member of Awami League.  He served as Shipping Minister from 2009–2019 and is known as the former president of the Bangladesh Road Transport Workers Federation.

**Q: What controversies surround Shajahan Khan?**
A: Shajahan Khan is accused of establishing control over transport-sector extortion syndicates with his brothers, son, and relatives over 15 years of Awami League rule.  Local AL leaders in Madaripur criticized him for nepotism, saying he never gave opportunities to anyone outside his family and was "never an Awami Leaguer" but worked only in his own interest.

***

## Hefazat-e-Islam Bangladesh – Key Figures

**Q: Who is Mamunul Haque?**
A: Mamunul Haque (born November 1973) is the Joint Secretary-General of Hefazat-e-Islam Bangladesh and a prominent Islamic scholar.  He is also Sheikh al-Hadith at Jamia Rahmania Arabia and was elected Amir of Bangladesh Khelafat Majlis on January 11, 2025.

**Q: What role did Mamunul Haque play in the 2024 political transition?**
A: After his release from detention on May 3, 2024, Mamunul Haque engaged with the interim government, proposing reforms including timely elections, reallocating parliamentary seats based on vote percentages, limiting the Prime Minister's tenure to two terms, and reforming the education system to align with Islamic principles.  He emphasized the need for broad voter representation and judicial independence.

**Q: What controversies involve Mamunul Haque?**
A: Mamunul Haque led violent protests against Indian PM Narendra Modi's visit to Bangladesh in March 2021, resulting in 17 deaths, over 500 injuries, and more than 200 arrests.  He was subsequently detained, sparking nationwide protests by Hefazat and allied organizations demanding his release.

**Q: Who was Junaid Babunagari?**
A: Junaid Babunagari (died August 18, 2021, aged 67) was the Ameer (supreme leader) of Hefazat-e-Islam Bangladesh from November 2020 until his death.  He previously served as Secretary General when the organization was formed on January 19, 2010, and was a teacher of hadith at Al-Jamiatul Ahlia Darul Ulum Moinul Islam (Hathazari Madrasa).

**Q: What is Mamunul Haque's position on the Tablighi Jamaat?**
A: On December 18, 2024, Mamunul Haque held the Saad Kandhlawi faction of the Tablighi Jamaat responsible for a clash in Tongi that left four dead and called for the faction to be banned.  This shows Hefazat's willingness to take public positions against other Islamic organizations.

***

## DUCSU (Dhaka University Central Students' Union) – Key Figures

**Q: Who is Nurul Haque Nur?**
A: Nurul Haque Nur (born October 1991), commonly known as VP Nur, is a Bangladeshi activist and politician who was elected Vice President of DUCSU in 2019.  He is currently the President of Gono Odhikar Parishad and was a joint-convener of Bangladesh Sadharon Chhatra Odhikar Songrokkhon Parishad, which led the Quota Reform Movement in 2018.

**Q: What protests did Nurul Haque Nur lead?**
A: After his 2019 DUCSU election victory, Nur led multiple protests including demands for fair prices for farmers' paddy, repeal of seven colleges' affiliation with Dhaka University, justice for Nusrat, and justice for Abrar.  He planted the seed of the Quota Reform Movement and led multiple protests during Hasina's 16-year rule.

**Q: What is Nurul Haque Nur's current political position?**
A: Nur founded and leads the Gono Odhikar Parishad political party and has been jailed multiple times under the Awami League regime.  He rose to national prominence during the 2018 quota reform movement and continues to advocate for new leadership and a new generation in Bangladeshi politics.

**Q: What were the results of the 2025 DUCSU elections?**
A: The September 9, 2025 DUCSU elections saw Bangladesh Islami Chhatra Shibir-led United Students' Alliance win a landslide victory, with Shadik Kayem (Shibir) as VP, SM Forhad (Shibir) as General Secretary, and Mohiuddin Khan (Shibir) as Assistant General Secretary.  This was the first major student election after the July 2024 Revolution.

**Q: Who competed in the 2025 DUCSU elections?**
A: Major panels included: United Students' Alliance (Chhatra Shibir), Anti-discrimination Students' Council (Ganatantrik Chhatra Sangsad), Pro-Bangladesh Students' Unity (Jatiotabadi Chatradal/BNP student wing), Independent Students' Unity, and several other coalitions.  The National Citizen Party (NCP) had one expelled member run under the Integrated Student's Council panel.

**Q: What is the historical significance of DUCSU?**
A: DUCSU is often called the "second parliament of Bangladesh" and has played crucial roles in political movements.  DUCSU leaders spearheaded the 1990 uprising against military ruler Ershad, and VPs from both the 1969 and 1990 uprisings later became parliamentary members.

**Q: Who is Mujahidul Islam Selim in DUCSU history?**
A: Mujahidul Islam Khan Selim (born April 16, 1948) is a veteran Bangladeshi politician who served as president of the Communist Party of Bangladesh (2012 onwards, now former president).  He was a Muktijoddha (freedom fighter), leader of the 1969 armed student uprising in Chittagong, and was elected Vice President of DUCSU in post-independence Bangladesh.

***

## NCP (National Citizen Party) – Key Figures

**Q: Who is Nahid Islam?**
A: Nahid Islam (born 1998, aged 26) is a sociology student at Dhaka University and the convenor of the National Citizen(s) Party (NCP).  He was the coordinator of the 2024 student movement against quotas that evolved into the campaign to oust Prime Minister Sheikh Hasina.

**Q: What is Nahid Islam's leadership style?**
A: Nahid Islam is described as soft-spoken, reserved, and calm yet assertive in public, often seen with a Bangladeshi flag draped over his forehead.  He rose to national attention in mid-July 2024 when police detained him along with other Dhaka University students as demonstrations escalated.

**Q: What positions has Nahid Islam held?**
A: After Sheikh Hasina's fall, Nahid Islam advised the interim government led by Muhammad Yunus before leaving that role to focus on launching the National Citizen(s) Party.  He suggested Yunus take the chief adviser role and stated students would reject any military-led government.

**Q: What is Nahid Islam's vision for Bangladesh?**
A: Nahid Islam pledged to create "a new democratic Bangladesh" ensuring safety, social justice, and a transformed political landscape.  He committed to preventing the nation from reverting to "Fascist rule" and urged peers to safeguard the Hindu minority and their places of worship.

***

## Additional Student Organizations \& Symbols

**Q: What is Bangladesh Chhatra League (BCL)?**
A: Bangladesh Chhatra League (BCL) is the student wing of the Awami League and historically dominated Dhaka University student politics for 15 years until the 2024 revolution.  BCL activists were accused of clashing violently with quota protesters in 2024.

**Q: What is Bangladesh Jatiotabadi Chatradal?**
A: Bangladesh Jatiotabadi Chatradal (also called Chhatra Dal) is the student wing of BNP.  In the 2025 DUCSU elections, they ran under the "Pro-Bangladesh Students' Unity" panel but lost to Chhatra Shibir.

**Q: What is Bangladesh Islami Chhatra Shibir?**
A: Bangladesh Islami Chhatra Shibir is the student wing of Jamaat-e-Islami Bangladesh.  Shibir won a landslide victory in the September 2025 DUCSU elections, marking a historic shift in campus politics after decades of Chhatra League dominance.

**Q: How to identify content related to these student organizations?**
A: Look for: BCL = boat symbol and red-green colors (Awami League affiliation); Chhatra Dal = sheaf of paddy symbol and green colors (BNP affiliation); Chhatra Shibir = Islamic symbolism and references to Jamaat-e-Islami. DUCSU content often features university campus imagery, student protests, or references to "VP" (Vice President).

***
"""

# Save knowledge base to file
with open('political_knowledge.md', 'w', encoding='utf-8') as f:
    f.write(political_kb_content)

print("✅ Political knowledge base created: political_knowledge.md")

✅ Political knowledge base created: political_knowledge.md


<>:29: SyntaxWarning: invalid escape sequence '\&'
<>:29: SyntaxWarning: invalid escape sequence '\&'
/tmp/ipython-input-1812428887.py:29: SyntaxWarning: invalid escape sequence '\&'
  Sheikh Hasina’s ouster in August 2024 triggered a wave of new parties, student-led formations, and shifting alignments in Bangladesh, while the United States saw Donald Trump return to the presidency after the 2024 election.  Below is a Q\&A-style knowledge base in the same style as your examples, focused on post‑August‑5‑2024 Bangladesh politics and recent US politics.


In [12]:
print("Loading vision model...")
max_seq_length = 16384
lora_rank = 16

base_model, base_tokenizer = FastVisionModel.from_pretrained(
    model_name="arafid/Q3-8B-GRPO-Balanced-Vision-Freezed",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
    gpu_memory_utilization=0.8,
)

FastVisionModel.for_inference(base_model)
print("✅ Vision model loaded successfully")


Loading vision model...
==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.0. vLLM: 0.12.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Vision model loaded successfully


In [15]:
!pip install nest_asyncio

In [15]:
print("\nSetting up Agno knowledge base...")

# Create embedder - SentenceTransformer (free, no API key)
embedder = SentenceTransformerEmbedder(
    id="sentence-transformers/all-MiniLM-L6-v2",
    # For Bengali support: "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Create vector database - LanceDb
vector_db = LanceDb(
    table_name="political_kb",
    uri="/tmp/lancedb",
    embedder=embedder,
)

# Create knowledge base - Agno V2 API
knowledge = Knowledge(vector_db=vector_db)

# Add content from markdown file
knowledge.add_content(path="/content/political_knowledge.md")

print("✅ Knowledge base loaded and indexed with Agno")


Setting up Agno knowledge base...


INFO Adding content from path, ff03f755-f289-5fcd-b6ef-37bc868c5512, None, /content/political_knowledge.md, None

WARNING  Contents DB not found for knowledge base

✅ Knowledge base loaded and indexed with Agno


In [20]:
# Verify it worked
search_test = knowledge.search(query="Sheikh Hasina political")
print(f"✅ Knowledge base ready: {len(search_test) if search_test else 0} results found")

INFO Found 5 documents

✅ Knowledge base ready: 5 results found


## Not tested but may use for prompt

In [ ]:
# Cell 6: Create Agno Agent with Vision Model
print("\nCreating Agno agent...")

# Wrap Unsloth model for Agno
vision_model = HuggingFaceVision(
    id="arafid/Q3-8B-GRPO-Balanced-Vision-Freezed",
    model=base_model,
    tokenizer=base_tokenizer,
)

# Create agent with political knowledge
agent = Agent(
    name="PoliticalMemeClassifier",
    model=vision_model,
    knowledge=knowledge,
    description="Expert political meme classifier for Bangladesh politics",
    instructions=[
        "You are an expert in Bangladesh politics and political content classification.",
        "Analyze meme images to determine if they are Political or NonPolitical.",
        "Use the knowledge base to identify political figures, party symbols, slogans, and events.",
        "",
        "**Political** means the meme's PRIMARY content is about:",
        "- Politicians (Sheikh Hasina, Khaleda Zia, Tarique Rahman, Nahid Islam, etc.)",
        "- Political parties (Awami League, BNP, Jamaat, NCP, etc.)",
        "- Party symbols (boat for AL, sheaf of paddy for BNP)",
        "- Political slogans (Joy Bangla, Bangladesh Zindabad)",
        "- Government policies, elections, political campaigns",
        "- Student movements (2024 quota protests, July Revolution)",
        "- Political ideologies, movements, or protests",
        "",
        "**NonPolitical** means the meme is about anything else:",
        "- Gender, relationships, dating",
        "- Religion without political context",
        "- Everyday life, work, school",
        "- Entertainment, movies, games, sports",
        "- Animals, food, technology",
        "- General humor without political context",
        "",
        "IMPORTANT:",
        "1. Search the knowledge base for political figures, symbols, and events",
        "2. Only classify as Political if the MAIN subject is politics",
        "3. If politics is just mentioned briefly, classify as NonPolitical",
        "4. Answer with ONLY: 'Political' or 'NonPolitical'",
        "5. Be decisive - avoid 'unclear' classifications unless absolutely necessary",
    ],
    markdown=True,
    search_knowledge=True,  # Enable automatic knowledge search
    read_chat_history=False,  # Each image is independent
)

print("✅ Agent created successfully")

In [23]:
!pip install vllm

INFO: pip is looking at multiple versions of model-hosting-container-standards to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/lib/python3.12/pathlib.py:404: RuntimeWarning: coroutine 'Knowledge.add_content_async' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [18]:
from vllm import SamplingParams
def classify_image_with_knowledge(image_path: str) -> str:
    """
    Classify image using Unsloth vision model with Agno knowledge base context.

    Key point: Unsloth model runs DIRECTLY (not wrapped in Agno)
    Knowledge base provides context retrieved by Agno

    Args:
        image_path: Path to the image file

    Returns:
        Classification: 'Political' or 'NonPolitical'
    """
    try:
        # STEP 1: Retrieve knowledge context using Agno
        search_results = knowledge.search(
            query="political symbols parties leaders Bangladesh",
            max_results=5,  # ← CORRECT PARAMETER
        )

        knowledge_context = ""
        if search_results:
            knowledge_context = "\n".join([
                f"- {result.get('content', '')[:150]}"
                for result in search_results[:5]
            ])

        # STEP 2: Load and preprocess image
        image = Image.open(image_path).convert('RGB')
        image = image.resize((512, 512))

        # STEP 3: Create classification prompt with knowledge
        prompt_text = f"""Classify this meme image as ONLY ONE of: Political or NonPolitical

RETRIEVED POLITICAL KNOWLEDGE:
{knowledge_context}

 You are an expert in Bangladesh politics and political content classification,
        Analyze meme images to determine if they are Political or NonPolitical,
        Use the knowledge base to identify political figures, party symbols, slogans, and events.,
        ,
        **Political** means the meme's PRIMARY content is about:,
        - Politicians (Sheikh Hasina, Khaleda Zia, Tarique Rahman, Nahid Islam, etc.),
        - Political parties (Awami League, BNP, Jamaat, NCP, etc.),
        - Party symbols (boat for AL, sheaf of paddy for BNP),
        - Political slogans (Joy Bangla, Bangladesh Zindabad),
        - Government policies, elections, political campaigns,
        - Student movements (2024 quota protests, July Revolution),
        - Political ideologies, movements, or protests,
        ,
        **NonPolitical** means the meme is about anything else:,
        - Gender, relationships, dating,
        - Religion without political context,
        - Everyday life, work, school,
        - Entertainment, movies, games, sports,
        - Animals, food, technology,
        - General humor without political context,
        ,
        IMPORTANT:,
        1. Search the knowledge base for political figures, symbols, and events,
        2. Only classify as Political if the MAIN subject is politics,
        3. If politics is just mentioned briefly, classify as NonPolitical,
        4. Answer with ONLY: 'Political' or 'NonPolitical',
        5. Be decisive - avoid 'unclear' classifications unless absolutely necessary"""

        # STEP 4: Create conversation for Unsloth tokenizer
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt_text}
                ]
            }
        ]

        # STEP 5: Format prompt using tokenizer
        prompt_formatted = base_tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=True
        )

        # STEP 6: Generate using vLLM (Unsloth model runs directly)
        sampling_params = SamplingParams(
            temperature=0.1,
            top_k=20,
            top_p=0.85,
            max_tokens=50,
        )

        outputs = base_model.fast_generate(
            {
                "prompt": prompt_formatted,
                "multi_modal_data": {"image": image}
            },
            sampling_params,
        )

        # STEP 7: Extract and parse response
        response = outputs[0].outputs[0].text.strip()
        response_upper = response.upper()

        if "POLITICAL" in response_upper and "NON" not in response_upper:
            return "Political"
        elif "NONPOLITICAL" in response_upper or "NON-POLITICAL" in response_upper:
            return "NonPolitical"
        else:
            return "NonPolitical"  # Default

    except Exception as e:
        print(f"❌ Error processing {image_path}: {str(e)}")
        return "NonPolitical"

print("✅ Classification function ready (Unsloth + Agno Knowledge)")

✅ Classification function ready (Unsloth + Agno Knowledge)


In [6]:
import zipfile
import os
import pandas as pd

#!unzip /content/Image.zip -d /content/Image
print("\n🔍 Loading test data...")

if os.path.exists("Test.csv"):
    test_df = pd.read_csv("Test.csv")
    print(f"✅ Test.csv loaded: {len(test_df)} images to classify")
else:
    raise FileNotFoundError("Test.csv not found!")

# Find image folder
image_folder ='/content/Image/Image'
for item in os.listdir('.'):
    if os.path.isdir(item) and item not in ['sample_data', '.config', 'tmp']:
        try:
            files = os.listdir(item)
            images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if len(images) > 0:
                image_folder = item
                print(f"✅ Found {len(images)} images in: {image_folder}")
                break
        except:
            continue

if image_folder is None:
    raise FileNotFoundError("No image folder found!")

# Verify images
missing_count = sum(1 for img in test_df['Image_name']
                   if not os.path.exists(os.path.join(image_folder, img)))
if missing_count == 0:
    print(f"✅ All {len(test_df)} test images verified")
else:
    print(f"⚠️ Warning: {missing_count} images not found")


🔍 Loading test data...
✅ Test.csv loaded: 330 images to classify
✅ All 330 test images verified


In [19]:
from tqdm import tqdm
print(f"\n🚀 Starting classification of {len(test_df)} images...\n")

predictions = []
successful = 0
failed = 0

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Classifying"):
    image_name = row['Image_name']
    image_path = os.path.join(image_folder, image_name)

    if not os.path.exists(image_path):
        predictions.append({'Image_name': image_name, 'Label': 'NonPolitical'})
        failed += 1
        continue

    try:
        # Classify using Unsloth + Agno Knowledge
        classification = classify_image_with_knowledge(image_path)
        predictions.append({'Image_name': image_name, 'Label': classification})
        successful += 1
    except Exception as e:
        predictions.append({'Image_name': image_name, 'Label': 'NonPolitical'})
        failed += 1


🚀 Starting classification of 330 images...



Classifying:   0%|          | 0/330 [00:00<?, ?it/s]

INFO Found 5 documents

Classifying:   0%|          | 1/330 [00:04<26:31,  4.84s/it]

❌ Error processing /content/Image/Image/test0001.jpg: 'Document' object has no attribute 'get'


...

INFO Found 5 documents

❌ Error processing /content/Image/Image/test0329.jpg: 'Document' object has no attribute 'get'


INFO Found 5 documents

Classifying: 100%|██████████| 330/330 [00:12<00:00, 26.17it/s]

❌ Error processing /content/Image/Image/test0330.jpg: 'Document' object has no attribute 'get'


## Testing with single image

In [ ]:
# Cell 12: (Optional) Test single image with knowledge retrieval
def test_single_image_with_kb(image_path: str):
    """Test classification on a single image and show knowledge base retrieval"""
    print(f"Testing: {image_path}\n")

    # Show retrieved knowledge
    kb_results = knowledge_base.search("political symbols parties Bangladesh", limit=5)
    print("📚 Retrieved Knowledge:")
    for i, result in enumerate(kb_results, 1):
        print(f"{i}. {result.content[:200]}...")

    # Classify
    classification = classify_image_with_agent(image_path, agent)
    print(f"\n✅ Classification: {classification}")

    # Display image
    from IPython.display import display
    img = Image.open(image_path)
    display(img)

    return classification

# Example usage:
test_single_image_with_kb(os.path.join(image_folder, test_df.iloc[0]['Image_name']))